# Covid montly data (2019-2022)

In [4]:
import os
import time
import pandas as pd
import pyalex
from pyalex import Works
from pathlib import Path
from tqdm import tqdm

# --- CONFIGURACIÓ ---
# 🆕 NO POSEM EL CORREU. Deixem que pyalex faci servir el mode anònim.
# Si tens una API Key, la pots posar a la línia de sota (recomanat per anar més ràpid)
# pyalex.config.api_key = "LA_TEVA_API_KEY" 

BATCH_SIZE = 50
DATA = Path("../data")
PROCESSED = DATA / "processed"

print("1. Carregant dades per identificar els DOIs...")
df_covid = pd.read_parquet(PROCESSED / "all_papers_enriched_covid.parquet")

# Agafem els DOIs únics per no fer peticions repetides
dois_unics = df_covid["doi"].dropna().unique().tolist()
print(f"Tenim {len(dois_unics)} DOIs únics per consultar.")

print("2. Consultant dates a OpenAlex...")
resultats_dates = []
batches = [dois_unics[i:i + BATCH_SIZE] for i in range(0, len(dois_unics), BATCH_SIZE)]

for batch in tqdm(batches, desc="Batches"):
    try:
        # Només demanem l'ID i la data de publicació
        res = (Works()
               .filter_or(doi=batch)
               .select(["ids", "publication_date"])
               .get(per_page=BATCH_SIZE))
        
        for work in res:
            doi_retornat = work.get("ids", {}).get("doi")
            data_pub = work.get("publication_date")
            
            if doi_retornat and data_pub:
                doi_net = doi_retornat.replace("https://doi.org/", "").lower()
                resultats_dates.append({"doi": doi_net, "publication_date": data_pub})
                
    except Exception as e:
        print(f"Error amb un lot: {e}")
        time.sleep(2) # petita pausa per si de cas

df_dates = pd.DataFrame(resultats_dates)
print(f"S'han obtingut dates per a {len(df_dates)} papers.")

print("3. Processant les dates i creuant dades...")
df_dates["publication_date"] = pd.to_datetime(df_dates["publication_date"])
df_dates["year_openalex"] = df_dates["publication_date"].dt.year
df_dates["month_openalex"] = df_dates["publication_date"].dt.month

df_final = df_covid.merge(df_dates, on="doi", how="left")

# Reanomenem o substituïm les columnes perquè siguin les d'OpenAlex
df_final["year"] = df_final["year_openalex"].fillna(df_final["year"]).astype(int)
df_final["month"] = df_final["month_openalex"].fillna(df_final["month"]).astype(int)

# Seleccionem exactament les mateixes columnes
columnes_finals = ["year", "month", "doi", "topic_name", "domain_name"]
df_final = df_final[columnes_finals]

# 🆕 COMPTADOR DE PAPERS NO TROBATS A OPENALEX
papers_no_trobats = df_final["year"].isna().sum() # Si l'any és NaN, és que no s'ha trobat
print(f"\n⚠️ ATENCIÓ: {papers_no_trobats} papers NO s'han trobat a OpenAlex.")
print("   Per a aquests papers, s'ha fet servir la data de la Generalitat com a reserva.")

print("4. Guardant el nou fitxer...")
output_path = PROCESSED / "all_papers_enriched_covid_months.parquet"
df_final.to_parquet(output_path, index=False)

print(f"\nFet! Fitxer desat a: {output_path}")
display(df_final.head())

1. Carregant dades per identificar els DOIs...
Tenim 88134 DOIs únics per consultar.
2. Consultant dates a OpenAlex...


Batches:   4%|▍         | 69/1763 [00:40<18:30,  1.52it/s]

Error amb un lot: 400 Client Error: Bad Request for url: https://api.openalex.org/works?filter=doi:10.1155%2F2019%2F8912390%7C10.14198%2Ffem.2019.33.13%7C10.1128%2Faac.01586-18%7C10.1016%2Fj.postharvbio.2019.110934%7C10.1016%2Fj.neuroscience.2018.10.030%7C10.3390%2Fw11112385%7C10.5209%2Frced.63394%7C10.1016%2Fj.marpetgeo.2019.02.014%7C10.3989%2Faespa.092.019.015%7C10.1007%2F978-1-4939-8793-1_22%2C%7C10.1088%2F1755-1315%2F222%2F1%2F012027%7C10.1371%2Fjournal.pone.0215511%7C10.1016%2Fj.agrformet.2019.01.039%7C10.1016%2Fj.neuropharm.2019.02.026%7C10.3390%2Fnu11010083%7C10.1016%2Fj.compchemeng.2019.01.016%7C10.1016%2Fj.ijmecsci.2018.11.012%7C10.1080%2F00344893.2019.1592013%7C10.1002%2Fchem.201902226%7C10.1016%2Fj.scitotenv.2019.133989%7C10.1093%2Fbrain%2Fawz031%7C10.1016%2Fj.disc.2018.08.020%7C10.1159%2F000503712%7C10.1109%2Ftla.2019.8863312%7C10.1038%2Fs41534-019-0161-z%7C10.1038%2Fs41586-019-1545-0%7C10.1215%2F03335372-7259859%7C10.3389%2Ffphys.2019.00688%7C10.1111%2Fmec.14963%7C10.1177%

Batches:  21%|██        | 374/1763 [03:42<12:47,  1.81it/s]

Error amb un lot: 400 Client Error: Bad Request for url: https://api.openalex.org/works?filter=doi:10.3389%2Ffmed.2019.00273%7C10.1186%2Fs13048-019-0591-4%7C10.3389%2Ffphy.2019.00194%7C10.1073%2Fpnas.1914395116%7C10.1080%2F02673037.2019.1566521%7C10.1103%2Fphysreve.100.053110%7C10.1021%2Facs.joc.9b02544%7C10.1161%2Fcirculationaha.119.040267%7C10.1021%2Facsomega.9b02706%7C10.1021%2Facs.biochem.9b00390%7C10.3989%2Fgladius.2019.02%7C10.3390%2Fijms20235950%7C10.1182%2Fbloodadvances.2019000922%7C10.5194%2Facp-19-14211-2019%7C10.3390%2Fe21121153%7C10.1088%2F1757-899x%2F700%2F1%2F012012%7C10.1088%2F1757-899x%2F700%2F1%2F012003%7C10.1103%2Fphysrevd.100.106014%7C10.32870%2Fap.v11n2.1598%7C10.1111%2F1462-2920.14877%7C10.1016%2Fj.envpol.2019.113680%7C10.1021%2Facs.nanolett.9b04187%7C10.1016%2Fj.marpetgeo.2019.104148%7C10.1111%2F1365-2664.13550%7C10.1075%2Fterm.00035.vaz%7C10.1371%2Fjournal.pone.0224573%7C10.1016%2Fj.medcli.2019.07.027%7C10.1016%2Fj.molliq.2019.112223%7C10.1002%2Fadvs.201902420%7C

Batches:  42%|████▏     | 737/1763 [07:21<09:31,  1.80it/s]

... (1,540 outputs left out)

Error amb un lot: HTTPSConnectionPool(host='api.openalex.org', port=443): Max retries exceeded with url: /works?filter=doi:10.1111%2F1755-0998.13320%7C10.1016%2Fj.rse.2020.112121%7C10.1007%2Fs40725-021-00140-z%7C10.1126%2Fscience.abg7484%7C10.51443%2Frestaurarios.2021.01%7C10.3390%2Fijerph18137019%7C10.1111%2Fnph.17461%7C10.1016%2Fj.tree.2021.06.015%7C10.1029%2F2021gl092933%7C10.21138%2Fgf.767%7C10.3354%2Fdao03631%7C10.1111%2Fnph.17667%7C10.1038%2Fs41598-021-95575-6%7C10.1007%2Fs10530-021-02673-7%7C10.1016%2Fj.agee.2021.107653%7C10.1103%2Fphysrevlett.126.048901%7C10.1093%2Fnsr%2Fnwab110%7C10.5194%2Fessd-13-5353-2021%7C10.1038%2Fs41559-021-01412-4%7C10.1016%2Fj.pecon.2021.03.005%7C10.1007%2Fs10265-021-01306-3%7C10.3390%2Fjof8010041%7C10.17979%2Fbac.2021.11.0.7358%7C10.47487%2Fapcyccv.v2i4.167%7C10.1016%2Fj.rec.2021.03.010%7C10.12957%2Fredp.2022.64373%7C10.30893%2Feq.v0i20.199%7C10.4000%2Fcea.6635%7C10.1007%2Fs00192-020-04555-3%7C10.48489%2Fquadrante.23878%7C10.1002%2Fuog.24977%7C10.2120

Batches: 100%|██████████| 1763/1763 [56:51<00:00,  1.93s/it]


S'han obtingut dates per a 48794 papers.
3. Processant les dates i creuant dades...

⚠️ ATENCIÓ: 0 papers NO s'han trobat a OpenAlex.
   Per a aquests papers, s'ha fet servir la data de la Generalitat com a reserva.
4. Guardant el nou fitxer...

Fet! Fitxer desat a: ..\data\processed\all_papers_enriched_covid_months.parquet


,year,month,doi,topic_name,domain_name
0,2018,10,10.1093/sleep/zsy205,Psychology,Social Sciences
1,2018,10,10.1109/tcomm.2018.2876295,Computer Science,Physical Sciences
2,2019,9,10.1016/j.jchf.2019.04.022,Medicine,Health Sciences
3,2018,11,10.1016/j.phrs.2018.11.032,"Biochemistry, Genetics and Molecular Biology",Life Sciences
4,2019,2,10.1111/jsr.12830,Medicine,Health Sciences


In [ ]:
import os
import time
import pandas as pd
import pyalex
from pyalex import Works
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv("../.env") # Carrega el fitxer .env
pyalex.config.api_key = os.environ["OPENALEX_API_KEY"] # La llegeix de l'entorn

BATCH_SIZE = 50
PROCESSED = Path("../data/processed")

print("1. Carregant dades...")
df_covid = pd.read_parquet(PROCESSED / "all_papers_enriched_covid.parquet")
dois_unics = df_covid["doi"].dropna().unique().tolist()

# 🆕 2. Comprovem quins ja hem consultat (si el fitxer de sortida existeix)
output_path = PROCESSED / "all_papers_enriched_covid_months.parquet"
dois_fets = set()
if output_path.exists():
    df_ja_fet = pd.read_parquet(output_path)
    # Si el fitxer té la columna year_openalex i no és buida, ja el tenim
    if "year_openalex" in df_ja_fet.columns:
        dois_fets = set(df_ja_fet[df_ja_fet["year_openalex"].notna()]["doi"].unique())
        print(f"Ja tenim {len(dois_fets)} DOIs consultats d'una execució anterior.")

# Filtrem la llista per només consultar els que ens falten
dois_pendents = [d for d in dois_unics if d not in dois_fets]
print(f"Ens falten {len(dois_pendents)} DOIs per consultar.")

# 3. Consulta (només dels pendents)
resultats_dates = []
batches = [dois_pendents[i:i + BATCH_SIZE] for i in range(0, len(dois_pendents), BATCH_SIZE)]

for batch in tqdm(batches, desc="Batches pendents"):
    try:
        res = (Works()
               .filter_or(doi=batch)
               .select(["ids", "publication_date"])
               .get(per_page=BATCH_SIZE))
        
        for work in res:
            doi_retornat = work.get("ids", {}).get("doi")
            data_pub = work.get("publication_date")
            if doi_retornat and data_pub:
                doi_net = doi_retornat.replace("https://doi.org/", "").lower()
                resultats_dates.append({"doi": doi_net, "publication_date": data_pub})
                
    except Exception as e:
        pass # Si falla, ho saltem (es farà servir la data de la Generalitat)

# 4. Processem els nous resultats i unim amb els antics
df_noves_dates = pd.DataFrame(resultats_dates)
if not df_noves_dates.empty:
    df_noves_dates["publication_date"] = pd.to_datetime(df_noves_dates["publication_date"])
    df_noves_dates["year_openalex"] = df_noves_dates["publication_date"].dt.year
    df_noves_dates["month_openalex"] = df_noves_dates["publication_date"].dt.month

# 5. Unim amb les dades originals
df_final = df_covid.merge(df_noves_dates, on="doi", how="left")

# Si veníem d'una execució anterior, recuperem les dades que ja havíem trobat
if output_path.exists() and "year_openalex" in df_ja_fet.columns:
    df_final = df_final.merge(df_ja_fet[["doi", "year_openalex", "month_openalex"]], 
                              on="doi", how="left", suffixes=("", "_antic"))
    # Omplim els buits amb les dades antigues
    df_final["year_openalex"] = df_final["year_openalex"].fillna(df_final["year_openalex_antic"])
    df_final["month_openalex"] = df_final["month_openalex"].fillna(df_final["month_openalex_antic"])

# Utilitzem OpenAlex si està disponible, si no la Generalitat
df_final["year"] = df_final["year_openalex"].fillna(df_final["year"]).astype(int)
df_final["month"] = df_final["month_openalex"].fillna(df_final["month"]).astype(int)

columnes_finals = ["year", "month", "doi", "topic_name", "domain_name"]
df_final = df_final[columnes_finals]

# 6. Guardem
df_final.to_parquet(output_path, index=False)

papers_no_trobats = df_final["year"].isna().sum()
print(f"\nFet! S'ha desat el fitxer a: {output_path}")
print(f"⚠️ ATENCIÓ: {papers_no_trobats} papers NO s'han trobat a OpenAlex.")
print("   Per a aquests papers, s'ha fet servir la data de la Generalitat com a reserva.")

1. Carregant dades...
Ens falten 88134 DOIs per consultar.


Batches pendents:  84%|████████▎ | 1474/1763 [19:24<04:09,  1.16it/s] 